### Importing Packages

In [50]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated
import json

### Tool Creation

In [17]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

### Tool Binding

In [72]:
modal = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [73]:
modal_tools = modal.bind_tools([
    get_conversion_factor,
    convert
])

In [74]:
messages = [
    HumanMessage('What is conversion factor between USD and PKR?, and based on that can convert 1 usd to pkr?')
]

In [75]:
messages

[HumanMessage(content='What is conversion factor between USD and PKR?, and based on that can convert 1 usd to pkr?', additional_kwargs={}, response_metadata={})]

In [76]:
ai_message = modal_tools.invoke(messages)

In [77]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'PKR'},
  'id': 'bkmktvw1t',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 1},
  'id': 'bybch8af4',
  'type': 'tool_call'}]

In [78]:
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [79]:
messages

[HumanMessage(content='What is conversion factor between USD and PKR?, and based on that can convert 1 usd to pkr?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1751068801, "time_last_update_utc": "Sat, 28 Jun 2025 00:00:01 +0000", "time_next_update_unix": 1751155201, "time_next_update_utc": "Sun, 29 Jun 2025 00:00:01 +0000", "base_code": "USD", "target_code": "PKR", "conversion_rate": 283.9276}', name='get_conversion_factor', tool_call_id='bkmktvw1t'),
 ToolMessage(content='283.9276', name='convert', tool_call_id='bybch8af4')]

In [80]:
modal_tools.invoke(messages).content

''